##### Widgets and parameters

In [0]:
dbutils.widgets.text("storage_account", "dlspl21databricks")
dbutils.widgets.text("sotrage_container", "yanquiel")
dbutils.widgets.text("secret_scope", "yanquiel-kv-scope")
dbutils.widgets.text("raw_folder", "orders")


storage_account = dbutils.widgets.get("storage_account")
storage_container = dbutils.widgets.get("sotrage_container")
secret_scope = dbutils.widgets.get("secret_scope")
raw_folder = dbutils.widgets.get("raw_folder")
mount_point = f"/mnt/{storage_container}"

##### Keys within this specific scope

In [0]:
dbutils.secrets.list(scope=secret_scope)

##### Read SPN credentials from the secret scope 

In [0]:
tenant_id = dbutils.secrets.get(scope=secret_scope, key="tenant-id")

config = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": dbutils.secrets.get(scope=secret_scope, key="sp-databricks-adls-appid"),
    "fs.azure.account.oauth2.client.secret": dbutils.secrets.get(scope=secret_scope, key="sp-databricks-adls-appkey"),
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}
print("OAuth config successfully built from secret scope")

##### Attempt the mount

In [0]:
try:
    dbutils.fs.mount(
        source=f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/",
        mount_point=mount_point,
        extra_configs=config
    )
    display(dbutils.fs.ls(mount_point))
except Exception as e:
    print(f"Mount not available in this workspace: {e}")

##### Actual data access 

In [0]:
raw_path = f"abfss://{storage_container}@{storage_account}.dfs.core.windows.net/raw/{raw_folder}/"
display(dbutils.fs.ls(raw_path))



##### Read the raw

In [0]:
df_orders = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_path)
display(df_orders.limit(10))